# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook guides you through exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library. We'll load metadata, review record sets and fields (referenced by their `@id`), extract tabular data, process records, and visualize results.

### Dataset Source
The data is described by a Croissant schema (JSON-LD format) available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Tags: {getattr(metadata, 'keywords', [])}\n")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields, and their `@id` values.

We'll print the list of record set `@id`s in this dataset, and for each record set, list the fields (and columns, if available), referencing all entities by their `@id` as required.

In [ ]:
# Helper to extract all record sets and their field @ids:
record_sets = []

for rs in dataset.record_sets:
    print(f"Record Set @id: {rs['@id']}")
    record_sets.append(rs['@id'])
    if 'field' in rs:
        print(" Fields:")
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)} (label: {field.get('name','')})")
            else:
                print(f"    - {field}")
    if 'column' in rs:
        print(" Columns:")
        cols = rs['column']
        if isinstance(cols, dict):
            cols = [cols]
        for col in cols:
            print(f"    - {col.get('@id', col)} (label: {col.get('name','')})")
    print()
if not record_sets:
    print("No record sets defined in the Croissant metadata.")
# For this dataset, manual inspection may be required: try listing available tables from the API.

# Optionally, enumerate available tables/record sets from the dataset
for record_set in dataset.available_record_sets():
    print(f"Record Set: {record_set}")

## 3. Data Extraction
We'll attempt to load data from each record set listed above, using their `@id` strings, into pandas DataFrames for further analysis.

If the dataset has no explicit record set entries in the metadata, we'll discover available record set IDs programmatically using the `available_record_sets()` method.

In [ ]:
# Retrieve all available record set IDs
record_set_ids = list(dataset.available_record_sets())
print("Available record sets:")
for idx, rs_id in enumerate(record_set_ids):
    print(f"  {idx+1}. {rs_id}")

dataframes = {}
for record_set in record_set_ids:
    print(f"\nLoading data for record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set].columns.tolist()}")
        print(dataframes[record_set].head(2).to_string(index=False))
    else:
        print("  No records found.")

# Choose the first non-empty record set for subsequent exploration
for rs in record_set_ids:
    if rs in dataframes and not dataframes[rs].empty:
        primary_record_set_id = rs
        break
print(f"\nSelected primary record set for further analysis: {primary_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's apply data processing techniques to the loaded DataFrame:
 - **Filtering records** (e.g., above/below a threshold value)
 - **Normalizing a numeric field**
 - **Grouping/categorizing by a key attribute

**All fields are referenced by their `@id`.**

In [ ]:
# Display the columns/fields with @id for the chosen record set
df = dataframes[primary_record_set_id]
print("Field (@id) Names for analysis:")
for col in df.columns:
    print(f" - {col}")

# Attempt to automatically select a numeric field by checking dtypes
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"\nSelected numeric field for EDA: {numeric_field_id}")
else:
    # fallback: pick any field
    numeric_field_id = df.columns[0]

# Threshold for filtering (example: 10 or median)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].quantile(0.75)
else:
    threshold = None

if threshold is not None:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (top quartile):")
    print(filtered_df.head().to_string(index=False))
    # Normalization
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm]].head().to_string(index=False))
else:
    print(f"No numeric fields available for filtering/normalization.\n")

# Now group by a categorical field (if one exists)
categorical_fields = df.select_dtypes(include=['object','category']).columns.tolist()
# Remove numeric_field_id from candidates
group_field = None
for c in categorical_fields:
    if c != numeric_field_id:
        group_field = c
        break
if group_field is not None and threshold is not None:
    grouped_df = filtered_df.groupby(group_field).agg({numeric_field_id: 'mean'})
    print(f"\nGrouped data by {group_field}, mean of {numeric_field_id}:")
    print(grouped_df.head().to_string())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and (if grouped) the group means.

All field labels shown are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=20)
plt.title(f"Distribution of field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# If grouped_df is available, plot group means
if 'grouped_df' in locals() and group_field is not None and threshold is not None:
    plt.figure(figsize=(8,4))
    grouped_df.sort_values(numeric_field_id, ascending=False).plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load and inspect a FAIR<sup>2</sup> dataset using the `mlcroissant` library
- Reference all dataset elements (record sets, fields) using their `@id` for transparency
- Extract tabular data for EDA, apply filtering, normalization, and grouping
- Visualize key numeric distributions and relationships

To extend this analysis:
- Examine other record sets (by their `@id`) using the same methodology
- Map field `@id`s to human-friendly labels for reports
- Apply your own machine learning or statistical models using the structured data

> **Note**: Always use `@id` for programmatic referencing, especially for reproducible data science workflows involving FAIR datasets.